# Priority Workloads and Top-K Selection


```{contents}
:local:
:depth: 2
```


```{index} Keep the Boundary of the Best K Items
```



```{index} Keep the Boundary of the Best K Items
```

## Keep the Boundary of the Best K Items



A sales report needs the largest three amounts from a stream. Sorting every amount is one possible solution, but the requested result is smaller than the full sorted input. A min-heap of the currently retained best three exposes the weakest retained amount at its root. That is exactly the boundary a new candidate must beat.

For amounts 7, 2, 9, 4, 10, retain 7, 2 and 9 first. Their minimum is 2. Candidate 4 replaces 2; candidate 10 replaces the new minimum 4. The retained values are 7, 9 and 10. Their heap storage need not be printed in sorted report order; sorting only the final retained values is a separate reporting step.

| Candidate | Decision | Retained values as a set of amounts |
| --- | --- | --- |
| 7, 2, 9 | Fill three places | 2, 7, 9 |
| 4 | Replace the smallest retained amount | 4, 7, 9 |
| 10 | Replace the smallest retained amount | 7, 9, 10 |

After each processed prefix, the heap contains the best min(k, processed count) entries under the chosen comparison policy. If a new candidate is no better than the root and the heap is full, at least k retained entries are already as good, so discarding it preserves the invariant. If it is better, replacing the weakest retained entry restores the best-k claim. This argument explains the algorithm rather than just observing a successful result.

Decide whether k zero is a valid request, what to do when fewer than k entries exist, and how tied amounts affect which record IDs are retained. Keeping amounts alone loses record identity. A report requiring deterministic tie order needs a secondary policy, not an assumption about library heap stability.

Priority also differs from fairness. A permanently urgent stream can delay ordinary requests indefinitely even when each minimum-priority removal is correct. Aging, reserved service opportunities or another explicit scheduling policy may be needed. That is a requirement choice layered on the structure, not a defect fixed by reversing the heap comparison.

Guided variation: select the smallest three amounts instead. Explain why a max-heap exposes the weakest retained boundary for that goal. Then retain complete invoice records with equal amounts and state what additional information your comparison must use if arrival order breaks ties.

For related explanation, see the further reading below.[^development-reading]


A priority queue is useful when the next item is chosen by urgency or score rather than arrival position. Use a clearly stated tie rule for service work and a clearly stated retained-set rule for top-k reporting.

The following applications use the library queue so you can focus on the workload. The heap implementation in the previous section explains the repair work behind this behavior. A dictionary-based cancellation index is introduced only after hash collections in the next chapter.


```{index} Top-K with a Min-Heap
```

## Top-K with a Min-Heap




```{index} top-k; min-heap; streaming
```

Keep the k largest observed transaction amounts without sorting the entire input. Maintain at most k retained values in a min-heap. The root is the smallest retained value, so it is the candidate to discard when a larger value arrives.

**Invariant:** after each input, the heap contains the largest `min(k, valuesSeen)` values of that prefix, including repeated values. If fewer than k values are stored, add the next value. Once full, a value no larger than the root cannot improve the retained multiset; a larger one replaces that root.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

int[] amounts = { 72, 91, 84, 99, 67, 88, 95, 95 };
int k = 3;
Console.WriteLine("top=" + string.Join(", ", TopK(amounts, k)));
Console.WriteLine($"zero-k count={TopK(amounts, 0).Count}");

List<int> TopK(IEnumerable<int> values, int k)
{
    if (k < 0) throw new ArgumentOutOfRangeException(nameof(k));
    if (k == 0) return new List<int>();
    var retained = new PriorityQueue<int, int>();
    foreach (int value in values)
    {
        if (retained.Count < k) retained.Enqueue(value, value);
        else if (value > retained.Peek())
        {
            retained.Dequeue();
            retained.Enqueue(value, value);
        }
    }
    var result = new List<int>();
    while (retained.TryDequeue(out int value, out _)) result.Add(value);
    result.Reverse();
    return result;
}

top=99, 95, 95
zero-k count=0


For k = 3, retained values after the first three inputs are 72, 84, 91. Processing 99 drops 72; 67 changes nothing; 88 drops 84; 95 drops 88; the second 95 drops 91. The final result is `99, 95, 95`. Repeated equal amounts remain separate observations.

For n inputs and positive k, maintenance has an O(n log(k + 1)) upper bound and O(k) storage, or fewer entries when n < k. Producing the final ordered result costs O(k log(k + 1)) in the worst case. k = 0 returns immediately; negative k is rejected. If record identity or tie order matters, include an explicit secondary priority and define which tied records to retain.

**Common mistake.** Using a max-heap for the k largest values. The root would then be the *best* retained value, not the one to discard, so every new candidate is compared against the wrong boundary; a min-heap exposes the weakest retained amount, which is the only one a candidate needs to beat.


```{index} Exercise
```

## Exercise




```{index} priority queue; exercise
```


Enqueue three tickets with priorities of your choice, where a lower number means more urgent, then dequeue and print them until the queue is empty. The output must list the tickets from most to least urgent regardless of the order you enqueued them.


In [ ]:
// Exercise: Triage tickets
// Enqueue three tickets with priorities. Lower number means more urgent.
// Print them in priority order.

using System;
using System.Collections.Generic;

var tickets = new PriorityQueue<string, int>();

// Your code starts here.



// Your code ends here.


In [1]:
// Solution
using System;
using System.Collections.Generic;

var tickets = new PriorityQueue<string, int>();
tickets.Enqueue("Password reset", 3);
tickets.Enqueue("Payment outage", 0);
tickets.Enqueue("New laptop request", 5);

while (tickets.Count > 0)
{
    Console.WriteLine(tickets.Dequeue());
}


Payment outage
Password reset
New laptop request


The output is `Payment outage`, `Password reset`, `New laptop request`: priorities 0, 3, and 5 in ascending order, regardless of the order the three calls to `Enqueue` appeared in. Enqueue them in a different order and the output does not change.


```{index} Priority Is Different from Fairness
```

## Priority Is Different from Fairness




```{index} Priority Is Different from Fairness
```


A strict priority rule may postpone low-priority work indefinitely when urgent work keeps arriving. Stable arrival ties solve ordering among equal priorities, not fairness between priority levels. Choose a business rule for escalation or bounded urgent service before claiming a scheduler is fair.

For top-k, state whether duplicates count as separate observations, what k=0 means, and how results are finally displayed. Maintaining a heap of at most k retained values costs O(k) storage and O(log k) repair per retained update; producing a sorted output is additional work.


```{index} Exercise
```

## Practice: Explain Priority and Fairness




```{index} top-k; exercise
```

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// Exercise: keep the two largest transaction amounts.
int[] amounts = { 70, 95, 82, 99, 88 };
var top = new PriorityQueue<int, int>();
// Your code starts here.
/* TODO */
// Your code ends here.
var result = new List<int>();
while (top.TryDequeue(out int amount, out _)) result.Add(amount);
result.Reverse();
Console.WriteLine(string.Join(", ", result));

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// Solution
int[] amounts = { 70, 95, 82, 99, 88 };
var top = new PriorityQueue<int, int>();
foreach (int amount in amounts)
{
    top.Enqueue(amount, amount);
    if (top.Count > 2) top.Dequeue();
}
var result = new List<int>();
while (top.TryDequeue(out int amount, out _)) result.Add(amount);
result.Reverse();
Console.WriteLine(string.Join(", ", result));

99, 95


The output is `99, 95`. The queue never held more than two values; each time a third arrived, `Dequeue` discarded the smallest retained amount. Draining a min-heap yields ascending order, which is why the list is reversed before printing.


```{rubric} Footnotes
```


[^1803_priority_applications-1]: Cancellation and replacement require an additional membership/version policy. The next chapter combines dictionary lookup with priority selection after both structures have been taught.


[^development-reading]: Further reading: [OpenDSA CS3, Heaps and Priority Queues](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/Heaps.html). The worked business scenarios and guided reasoning here are original teaching examples.
